In [1]:
from string import Template
import json

import requests
import pandas as pd
import ast
import re

In [2]:
_temperature = 0
_max_tokens = -1
_stream = False

In [11]:
# Extraction of AI-specific requirements in documents
ai_specific_prompt_template = """
EXTRACT information from the following json input delimitated by ```. I want you to extract tasks on the key "AI_SPECIFIC".\
0. ALWAYS FINISH THE OUTPUT. Never send partial responses. JUST RETURN JSON TEXT.
1. The text input is part of an international standard document associated with the development lifecycle of AI systems. 
2. The input is in json format. The input looks like:
{
    "PROCESS_ID":string,
    "PROCESS_NAME":string,
    "PROCESS_2_ID":string,
    "PROCESS_2_NAME":string,
    "ACTIVITY":string,
    "ACTIVITY_ID":string,
    "AI_SPECIFIC":string
}
3. Generate tasks based on the json key AI_SPECIFIC from input, based on NOUNS and ACTIONS, where NOUNS and ACTIONS should be unique. NOUNS are assets as outputs of ai-specific, and ACTIONS are verbs associated to those NOUNS. Keep original text, do not modify it. Do not include two ACTIONS in one TASK. 
4. The output has a json format. The output has a key called AI_SPECIFIC_TASKS which contans a list of all tasks extracted from input json key AI_SPECIFIC. The extracted task save it in key "description". Create a short name and deposit in the key "name" that is the name of the task and must be short and must NOT contain ACTIONS (e.g., VERBS). The template includes ..., where it means more than one task. The output must should look like:\
{
    "AI_SPECIFIC_TASKS": [
        {"task_id":1,"name":string,"description":string},
        {"task_id":2,"name":string,"description":string},
        ...,
        {"task_id":n,"name":string,"description":string}
    ]
}


Input: 
```
$ctext
```
"""

In [12]:
# Extraction of AI-specific requirements in documents
ai_specific_prompt_template = """
EXTRACT information from the following json input delimitated by ```. I want you to extract tasks on the key "AI_SPECIFIC".
0. ALWAYS FINISH THE OUTPUT. Never send partial responses. JUST RETURN JSON TEXT. When refering to json, use the general tempalte => '{key:value}'
1. The text input is part of an international standard document, associated with the development lifecycle of AI systems. 
2. The input is in json format. The following is the template. The input should look like:
{
    "AI_SPECIFIC":string
}

4. Generate tasks based on the json key AI_SPECIFIC from input, based on NOUNS and ACTIONS, where NOUNS and ACTIONS should be unique. NOUNS are assets as outputs of ai-specific, and ACTIONS are verbs associated to those NOUNS. Keep original text, do not modify it. Do not include two ACTIONS in one TASK. 
5. The output must have a json format. The output has the key AI_SPECIFIC_TASKS which contans a list of tasks extracted from input AI_SPECIFIC. The extracted task save it in value of key "description". Create a short name and deposit in value of key "name" which represents the name of the task and must be short and must NOT contain ACTIONS (e.g., VERBS). The output must should look like, where "...," indicates that there can me more than one task in AI_SPECIFIC json input:\
{
    "AI_SPECIFIC_TASKS": [
        {"task_id":1,"name":string,"description":string},
        {"task_id":2,"name":string,"description":string},
        ...,
        {"task_id":n,"name":string,"description":string}
    ]
}


Input: 
```
$ctext
```
"""

In [41]:
# Extraction of AI-specific requirements in documents
ai_specific_prompt_template = """
EXTRACT information from the following json input delimitated by ```. I want you to extract tasks on the key "AI_SPECIFIC".  
0. ALWAYS FINISH THE OUTPUT. Never send partial responses. Just send JSON format.
1. The text input is part of an international standard document, associated with the development lifecycle of AI systems. 
2. The input is in json format. The following is the template. The input should look like:
[{
    "PROCESS_ID":string,
    "PROCESS_NAME":string,
    "PROCESS_2_ID":string,
    "PROCESS_2_NAME":string,
    "ACTIVITY":string,
    "ACTIVITY_ID":string,
    "AI_SPECIFIC":string

}]

with more than one element, with unique activity IDs.\

3. EXTRACT tasks based on the json input key AI_SPECIFIC. Extract tasks based on OBJECTS and VERBS in each sentence. A task must contain unique OBJECTS and VERBS. A OBJECTS is a goal/asset/outcome of an ACTIVITY and a VERB is an action associated to a OBJECT. Keep original task text in key "task_description" in the json output. Add another key "task_name" and create a name for the task based on the key "task_description".  Do not include two VERBS in one TASK. The output have json format, adding a list of tasks based on the key "AI_SPECIFIC". A new key should be added, named "AI_SPECIFIC_TASKS". The output must should look like:
{    "AI_SPECIFIC_TASKS": [
    {"task_id":1,"task_description":string,"task_name":string,},
    {"task_id":2,"task_description":string,"task_name":string,},
    ...,
    {"task_id":n,"task_description":string,"task_name":string,}
    ],
}
where "...," in output template means more than one task. \
    The output contains a key called AI_SPECIFIC_TASKS that contans a list of all tasks extracted from input json key AI_SPECIFIC. The name of the task must be short but descriptive enough and must NOT contain ACTIONS (e.g., VERBS). \

Input: 
```
$ctext
```
"""

In [52]:
# Extraction of AI-specific requirements in documents
ai_specific_prompt_template = """
EXTRACT information from the following json input delimitated by ```. I want you to extract tasks on the key "AI_SPECIFIC".  
0. ALWAYS FINISH THE OUTPUT. Never send partial responses. Just send JSON format.
1. The text input is part of an international standard document, associated with the development lifecycle of AI systems. 
2. The input is in json format. The JSON input looks like:
[{
    "PROCESS_ID":string,
    "PROCESS_NAME":string,
    "PROCESS_2_ID":string,
    "PROCESS_2_NAME":string,
    "ACTIVITY":string,
    "ACTIVITY_ID":string,
    "AI_SPECIFIC":string

}]

with more than one element, with unique activity IDs.\

3. EXTRACT tasks based on the json input key AI_SPECIFIC. \
Extract tasks based on OBJECTS and VERBS in each sentence. A task must contain a unique pair OBJECT and VERB. \
A OBJECTS is a goal/asset/outcome of an ACTIVITY and a VERB is an action associated to a OBJECT. \
A VERB refers to the action that should be performed and is interpreted as a set with the OBJECT to be acted on. \
An OBJECT is the obejct of a VREB and is interepreted as a set with the verb. \
Keep original task text in key "task_description" in the json output. \
Add another key "task_name" and create a name for the task based on the key "task_description".  Do not include two VERBS in one TASK. \
The output have json format, adding a list of tasks based on the key "AI_SPECIFIC". A new key should be added, named "AI_SPECIFIC_TASKS". A VERB refers to the action that should be performed and is interpreted as a set with the OBJECT to be acted on. An OBJECT is the obejct of a VREB and is interepreted as a set with the verb. The output must should look like:
{    "AI_SPECIFIC_TASKS": [
    {"task_id":1,"task_name":string,"task_description":string,},
    {"task_id":2,"task_name":string,"task_description":string,},
    ...,
    {"task_id":n,"task_name":string,"task_description":string,}
    ],
}
where "...," in output template means more than one task. \
    The output contains a key called AI_SPECIFIC_TASKS that contans a list of all tasks extracted from input json key AI_SPECIFIC. The name of the task must be short but descriptive enough and must NOT contain ACTIONS (e.g., VERBS). \

Input: 
```
$ctext
```
"""

In [32]:
# Extraction of AI-specific requirements in documents
ai_specific_prompt_template = """
EXTRACT information from the following json input delimitated by ```. I want you to extract tasks on the key "AI_SPECIFIC".  
0. ALWAYS FINISH THE OUTPUT. Never send partial responses. Just send JSON format.
1. The text input is part of an international standard document, associated with the development lifecycle of AI systems. 
2. The input is in json format. The JSON input looks like:
[{  
    "process_3_id_5338":string,
    "process_3_name_5338":string,
    "oat":string,
    "ai-specific":string

}]

with more than one element, with unique activity IDs.\

3. EXTRACT activities and tasks based on the json input keys "oat" and "ai-specific". \
An activity is a construct of related tasks for producing outcomes, Activities provide a means to organize related tasks within  the process,  to improve understanding and communication of  the process. \
An task is a detailed provision for implementation of a process. It may be presented as a requirement ('shall'), a recommendation ('should') or a permission ('may'). \
Extract activities and tasks based on OBJECTS and VERBS in each sentence, containing a unique pair OBJECT and VERB. \
An OBJECTS is a goal/asset/outcome of an ACTIVITY and a VERB is an action associated to a OBJECT. \
A VERB refers to the action that should be performed and is interpreted as a set with the OBJECT to be acted on. \
An OBJECT is the subject of a VERB and is interepreted as a set with the verb. \
When extracting tasks, keep original text in key in "task_description" in the json output. \
Add another key "activity_name" and "task_name", creating a name for the task based on the key "task_description".  Do not include two VERBS in one TASK. \
The output have json format, adding a list of tasks based on the key "AI_SPECIFIC". A new key should be added, named "AI_SPECIFIC_TASKS". A VERB refers to the action that should be performed and is interpreted as a set with the OBJECT to be acted on. An OBJECT is the obejct of a VREB and is interepreted as a set with the verb. \
The output must should look like the following:
{"AI_SPECIFIC_ACTIVTIES": 
 [
	{"activity_id":string,
	 "activity_name":string,
	 "tasks":[{"task_id":int,"task_name":string,"task_description":string}]}
 ]
}
The output contains a key called AI_SPECIFIC_ACTIVTIES that contans a list of activities and tasks extracted from input json. There could be more than one activity_id and task_id. The name of the activity and task must be short but descriptive enough and must NOT contain VERBS. The keys process_3_name_5338 and process_3_id_5338 in json input are not an activity \

Input: 
```
$ctext
```
"""

In [3]:
# Extraction of AI-specific requirements in documents
ai_specific_prompt_template = """
EXTRACT information from the following json input delimitated by ```. I want you to extract tasks on the key "AI_SPECIFIC".  
0. ALWAYS FINISH THE OUTPUT. Never send partial responses. Just send JSON format.
1. The text input is part of an international standard document, associated with the development lifecycle of AI systems. 
2. The input is in json format. The JSON input looks like:
[{  
	"task":string
}]

with more than one element, with unique activity IDs.\

3. EXTRACT tasks based on the json input key "task". \
An task is a detailed provision for implementation of a process. It may be presented as a requirement ('shall'), a recommendation ('should') or a permission ('may'). \
Extract tasks based on OBJECTS and VERBS in each sentence, containing a unique pair OBJECT and VERB. \
An OBJECTS is a goal/asset/outcome of an TASK and a VERB is an action associated to a OBJECT. \
A VERB refers to the action that should be performed and is interpreted as a set with the OBJECT to be acted on. \
An OBJECT is the subject of a VERB and is interepreted as a set with the verb. \
When extracting tasks, keep original text in key in "task_description" in the json output. \
Add another key "task_name", creating a name for the task based on the key "task_description".  Do not include two VERBS in one TASK. \
The output have json format, adding a list of tasks based on the key input "task". A new key should be added, named "TASKS". \
The output must should look like the following:
{"TASKS": 
 [{"task_id":int,"task_name":string,"task_description":string}]
}
The output contains a key called TASKS that contans a list of activities and tasks extracted from input json. There could be more than one task_id. \
Please, just send the JSON Output, do not add more text. 

Input: 
```
$ctext
```
"""

In [4]:
def send_request(message, system_msg="Always answer in rhymes."):
    # Your request payload
    url = "http://localhost:1234/v1/chat/completions"

    payload = {
        "messages": [
            { "role": "system", "content":  system_msg},
            { "role": "user", "content": message }
        ],
        "temperature": _temperature,
        "max_tokens": _max_tokens,
        "stream": _stream
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, data=json.dumps(payload))

    # Check for successful response
    if response.status_code == 200:
        return response.json()
    else:
        return {"error": "Request failed with status code " + str(response.status_code)}

In [5]:
# Function to take folder of files and a prompt template, and return a json-object of all the entities and relationships
def extract_ai_task(prompt_template, text, index):
    system_msg = "You are a helpful computer scientist and ISO standards expert who EXTRACTS information from documents."
    print(f"Extracting tasks {index}....")
    result = ""
    try:
        prompt = Template(prompt_template).substitute(ctext=text)
        result = send_request(prompt, system_msg=system_msg)
    except Exception as e:
        print(f"Error processing: {e}")
    return result

In [6]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = '62304_5338 (activity level).xlsx'
url_excel = f'../{excel_file_path}'

# Export the DataFrame to Excel
df = pd.read_excel(url_excel, sheet_name='5338_raw_guidelines')

In [7]:
df.head(2)

,id,process_3_id,process_3_name,ai_specific,ai_stage
0,4,6.4.7,Knowledge acquisition process,The project shall implement the following act...,NaN


In [54]:
df.drop(["id", "process_3_id",  "process_3_name", "process_2_id",  "process_2_name", "activity", "type_process_3", "activity_id", "activity","ai_stage"], axis='columns', inplace=True)

In [8]:
df.drop(["id", "process_3_id",  "process_3_name", "ai_stage"], axis='columns', inplace=True)

In [9]:
# Remove column 'B' from DataFrame
#df = df.drop(columns=['INDEX', 'PURPOSE', 'NOTE', 'OAT'])

In [9]:
json_data = json.loads(df.to_json(orient='records'))

In [10]:
json_data[0]

{'ai_specific': ' The project shall implement the following activities in accordance with applicable organization policies and procedures with respect to the knowledge acquisition process: a) Define scope and criteria for knowledge acquisition. As a first step, the scope and criteria of the knowledge are defined: What domain and what aspect is the knowledge about? How recent is the knowledge? b) Seek and gather sources of knowledge. Knowledge can be extracted from publications, data or experts. c) Perform knowledge acquisition to extract knowledge. In order to utilize the knowledge, publications can be studied, data can be analysed and experts interviewed or observed. In case of knowledge engineering, the extracted knowledge should be formalized in such a way that the algorithms involved can utilize it. This is part of the implementation process (see 6.4.9). d) Knowledge about the domain and the problem is gathered through studying, interviewing or other knowledge elicitation, data ana

In [11]:
response = extract_ai_task(ai_specific_prompt_template,json_data[0], 1)

Extracting tasks 1....


In [12]:
print(response["choices"][0]['message']['content'])


{
"TASKS": [
{
"task_id": 1,
"task_name": "Define scope and criteria for knowledge acquisition",
"task_description": "What domain and what aspect is the knowledge about? How recent is the knowledge?"
},
{
"task_id": 2,
"task_name": "Seek and gather sources of knowledge",
"task_description": "Knowledge can be extracted from publications, data or experts."
},
{
"task_id": 3,
"task_name": "Perform knowledge acquisition to extract knowledge",
"task_description": "In order to utilize the knowledge, publications can be studied, data can be analysed and experts interviewed or observed."
},
{
"task_id": 4,
"task_name": "Manage the result of knowledge acquisition",
"task_description": "NOTE 1 The roles, activities, constructional layers, components of knowledge engineering and their relationships, as well as a common knowledge engineering vocabulary are provided in ISO/IEC 5392[4]. NOTE 2 Knowledge collected for a project can also be shared between projects via repositories (areas storing know

In [63]:
responses = []
print(f"Running pipeline for {len(json_data)} items in ISO International Standard")
for index, json_item in enumerate(json_data):
    responses.append(extract_ai_task(ai_specific_prompt_template,json_item, index+1))

Running pipeline for 2 items in ISO International Standard
Extracting tasks 1....
Extracting tasks 2....


In [58]:
responses

[{'id': 'chatcmpl-u1vjojujjb7m5v8gnbv79',
  'object': 'chat.completion',
  'created': 1710235965,
  'model': '..\\.cache\\lm-studio\\models\\TheBloke\\Llama-2-13B-chat-GGUF\\llama-2-13b-chat.Q5_K_M.gguf',
  'choices': [{'index': 0,
    'message': {'role': 'assistant',
     'content': '\nPlease provide the extracted tasks in JSON format as specified in the instructions.\n\nNote: I have removed the irrelevant information from the input and output. The output should be in JSON format with the keys "TASKS" and "task_id", "task_name", and "task_description".'},
    'finish_reason': 'stop'}],
  'usage': {'prompt_tokens': 63,
   'completion_tokens': 63,
   'total_tokens': 126}},
 {'id': 'chatcmpl-j33r1d7iix98debgwhushv',
  'object': 'chat.completion',
  'created': 1710236284,
  'model': '..\\.cache\\lm-studio\\models\\TheBloke\\Llama-2-13B-chat-GGUF\\llama-2-13b-chat.Q5_K_M.gguf',
  'choices': [{'index': 0,
    'message': {'role': 'assistant',
     'content': '\nPlease provide the extracted t

### Reopen the conent in the txt file

In [3]:
file_path = "LM responses/responses_llm_11-03-2024.txt"  # Replace "example.txt" with the path to your text file

# Open the file in read mode
with open(file_path, "r") as file:
    # Read the contents of the file
    file_contents = file.read()

In [9]:
responses = file_contents

In [13]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = '62304_5338 (activity level).xlsx'
url_excel = f'../{excel_file_path}'

# Export the DataFrame to Excel
df = pd.read_excel(url_excel, sheet_name='5338_raw_guidelines')

In [14]:
column_names_5338 = df.columns.tolist()
column_names_5338.append("task_id")
column_names_5338.append("task_name")
column_names_5338.append("task_description")
print(column_names_5338)

['id', 'process_3_id', 'process_3_name', 'ai_specific', 'ai_stage', 'task_id', 'task_name', 'task_description']


In [46]:
regex = re.compile("Please provide the extracted tasks in JSON format as specified")
    
# Find all matches in the text
matches = regex.findall(responses)
print(matches)

['Please provide the extracted tasks in JSON format as specified',
 'Please provide the extracted tasks in JSON format as specified']

In [43]:
len(json.loads(responses))

13

In [44]:
json.loads(responses)[4]

{'id': 'chatcmpl-qx5gazjvghinw6qke104',
 'object': 'chat.completion',
 'created': 1710169874,
 'model': '..\\.cache\\lm-studio\\models\\TheBloke\\Llama-2-13B-chat-GGUF\\llama-2-13b-chat.Q5_K_M.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '\nPlease provide the extracted tasks in JSON format as specified in the instructions.\n\nNote: I have removed any unnecessary information from the input and output to make it easier to read and understand.'},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 39, 'completion_tokens': 39, 'total_tokens': 78}}

In [16]:
df_5338_with_tasks = pd.DataFrame(columns=column_names_5338)

for index, response in enumerate(json.loads(responses)):
    # Actually row in standard
    row = df.iloc[index]

    # Pattern to match inner JSON objects
    pattern = r'({\s*[^{}]+\s*})'

    # Find all matches
    matches = re.findall(pattern, response['choices'][0]['message']['content'], re.DOTALL)
    print(matches)
    #for match in matches:
    json_m = json.loads(response['choices'][0]['message']['content'])
    
    for task in json_m['TASKS']:
        new_row = row.copy().to_dict()
        new_row['task_id'] = task['task_id']
        new_row['task_name'] = task['task_name']
        new_row['task_description'] = task['task_description']
        new_row = pd.DataFrame([new_row])
        df_5338_with_tasks = pd.concat([df_5338_with_tasks, new_row], )
        


TypeError: the JSON object must be str, bytes or bytearray, not dict

In [18]:
df_5338_with_tasks = pd.DataFrame(columns=column_names_5338)

# Actually row in standard
row = df.iloc[0]

# Pattern to match inner JSON objects
pattern = r'({\s*[^{}]+\s*})'

# Find all matches
matches = re.findall(pattern, response['choices'][0]['message']['content'], re.DOTALL)
print(matches)
#for match in matches:
json_m = json.loads(response['choices'][0]['message']['content'])

for task in json_m['TASKS']:
    new_row = row.copy().to_dict()
    new_row['task_id'] = task['task_id']
    new_row['task_name'] = task['task_name']
    new_row['task_description'] = task['task_description']
    new_row = pd.DataFrame([new_row])
    df_5338_with_tasks = pd.concat([df_5338_with_tasks, new_row], )
        


['{\n"task_id": 1,\n"task_name": "Define scope and criteria for knowledge acquisition",\n"task_description": "What domain and what aspect is the knowledge about? How recent is the knowledge?"\n}', '{\n"task_id": 2,\n"task_name": "Seek and gather sources of knowledge",\n"task_description": "Knowledge can be extracted from publications, data or experts."\n}', '{\n"task_id": 3,\n"task_name": "Perform knowledge acquisition to extract knowledge",\n"task_description": "In order to utilize the knowledge, publications can be studied, data can be analysed and experts interviewed or observed."\n}', '{\n"task_id": 4,\n"task_name": "Manage the result of knowledge acquisition",\n"task_description": "NOTE 1 The roles, activities, constructional layers, components of knowledge engineering and their relationships, as well as a common knowledge engineering vocabulary are provided in ISO/IEC 5392[4]. NOTE 2 Knowledge collected for a project can also be shared between projects via repositories (areas sto

In [20]:
df_5338_with_tasks

,id,process_3_id,process_3_name,ai_specific,ai_stage,task_id,task_name,task_description
0,4,6.4.7,Knowledge acquisition process,The project shall implement the following act...,NaN,1,Define scope and criteria for knowledge acquis...,What domain and what aspect is the knowledge a...
0,4,6.4.7,Knowledge acquisition process,The project shall implement the following act...,NaN,2,Seek and gather sources of knowledge,"Knowledge can be extracted from publications, ..."
0,4,6.4.7,Knowledge acquisition process,The project shall implement the following act...,NaN,3,Perform knowledge acquisition to extract knowl...,"In order to utilize the knowledge, publication..."
0,4,6.4.7,Knowledge acquisition process,The project shall implement the following act...,NaN,4,Manage the result of knowledge acquisition,"NOTE 1 The roles, activities, constructional l..."


In [21]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = 'output-5338_task_2.xlsx'

# Export the DataFrame to Excel
df_5338_with_tasks.to_excel(excel_file_path, index=False)

In [46]:
#with open("responses_llm_11-03-2024.txt", 'w') as f:
#    f.write(json.dumps(responses))